# Recommender systems: Surprise's SVD, SVD++, NMF; Analytical SVD from scratch, GD SVD from scratch

## Libraries and constants

In [35]:
import gc
import os

from pathlib import Path

import numpy as np
import pandas as pd
from utils import get_null_info

from surprise import Dataset, Reader, SVD, SVDpp, NMF, accuracy, prediction_algorithms
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise.model_selection import GridSearchCV as surprise_GridSearchCV

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
# --- CONSTANTS ---
RANDOM_SEED = 42
n_jobs = os.cpu_count()-1

## Data import

In [6]:
links_raw = pd.read_csv(Path('data') / 'links.csv')
print(f"shape = {links_raw.shape}")
display(links_raw)

shape = (9742, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
9737,193581,5476944,432131.0
9738,193583,5914996,445030.0
9739,193585,6397426,479308.0
9740,193587,8391976,483455.0


The mapping of movie id to imdb and tmdb, I don't see any use in this dataset.

In [7]:
del links_raw
gc.collect()

164

In [3]:
movies_raw = pd.read_csv(Path('data') / 'movies.csv')
print(f"shape = {movies_raw.shape}")
display(movies_raw)

shape = (9742, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


Maps a movie id to its title and genres, can be useful to map the final predictions.

In [4]:
ratings_raw = pd.read_csv(Path('data') / 'ratings.csv')
print(f"shape = {ratings_raw.shape}")
display(ratings_raw)

shape = (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


The main dataframe with movies, users and ratings

In [10]:
tags_raw = pd.read_csv(Path('data') / 'tags.csv')
print(f"shape = {tags_raw.shape}")
display(tags_raw)

shape = (3683, 4)


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


Interesting moments in every movie with timestamps, we don't need it.

In [11]:
del tags_raw
gc.collect()

0

## EDA

In [5]:
ratings_raw.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


I don't see why we'd need the timestamp

In [6]:
ratings_raw = ratings_raw.drop(columns=['timestamp'])

In [7]:
duplicates_rating = ratings_raw[ratings_raw.duplicated()]
duplicates_rating

,userId,movieId,rating


In [8]:
ratings_raw.dtypes

userId       int64
movieId      int64
rating     float64
dtype: object

In [9]:
get_null_info(ratings_raw)

No missing values are found in the data_frame


""


In [10]:
ratings_raw['rating'].value_counts()

rating
4.0    26818
3.0    20047
5.0    13211
3.5    13136
4.5     8551
2.0     7551
2.5     5550
1.0     2811
1.5     1791
0.5     1370
Name: count, dtype: int64

It seems we are presented with a perfect data already obtained from the pivot table with only relevant data.

## Train (surprise library) - Task 1

### Prepare the data, train the baseline (SVD)

In [11]:
# Tell the surprise library the rating scale
scale = (ratings_raw.rating.min(), ratings_raw.rating.max())
reader = Reader(rating_scale=scale)

# Hand over the 𝒦: ONLY the three columns, in the order (user, item, rating).
data = Dataset.load_from_df(ratings_raw[['userId', 'movieId', 'rating']], reader)

# Split into train and test
df_train, df_test = surprise_train_test_split(data, test_size=0.2, random_state=RANDOM_SEED, shuffle=True)

# The stochastic gradient-based SVD
baseline_SVD = SVD(
    n_factors=100,              # k  — how many components to create to explain a user and an item entity
    n_epochs=20,                # how many times to loop through train
    lr_all=0.005,               # α  — learning rate to update all parameters
    reg_all=0.02,               # λ  — regularization term for all parameter updates
    biased=True,                # do use biases
    random_state=RANDOM_SEED,   # fixed random seed for reproducibility
)

# Train
baseline_SVD.fit(df_train)

In [12]:
def evaluate(model: prediction_algorithms, df_train: pd.DataFrame, df_test: pd.DataFrame):
    # Evaluate train
    train_tuples = df_train.build_testset()         # because surprise parses the train dataset separately for efficiency, we have to convert it back
    train_predictions = model.test(train_tuples)
    rmse_train = accuracy.rmse(train_predictions, verbose=False)

    # Evaluate test
    predictions = model.test(df_test)
    rmse_test = accuracy.rmse(predictions, verbose=False)

    print(f"RMSE (train) : {rmse_train:.4f}")
    print(f"RMSE (test) : {rmse_test:.4f}")

In [13]:
evaluate(baseline_SVD, df_train, df_test)

RMSE (train) : 0.6354
RMSE (test) : 0.8807


### Tuning with GridSearch and cross-validation

In [14]:
# 1. Create grid search training grid
grids = {
    'SVD': {
        'algo_class': SVD,
        'param_grid': {
            'n_factors': [50, 100],      # k — number of latent factors
            'n_epochs':  [20, 30],       # passes over the data
            'lr_all':    [0.005, 0.01],  # α — learning rate
            'reg_all':   [0.02, 0.1],    # λ — regularization
        },
    },
    'SVD++': {
        'algo_class': SVDpp,
        'param_grid': {
            'n_factors': [20, 30],
            'n_epochs': [10],
            'lr_all':    [0.005],
            'reg_all':   [0.02],
        },
    },
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [10, 20],       # NMF usually requires fewer factors
            'n_epochs': [15],
            # NMF has a separate regularization for users and items
            'reg_pu':    [0.06],
            'reg_qi':    [0.06],
        },
    },
}

best_models = {}

# 2. Run grid search
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=3,
        n_jobs=n_jobs,
    )
    gs.fit(data)

    # Save results
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Best algorithm
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"Best algorithm: {winner}")
print(f"RMSE: {best_models[winner]['score']}")
print(f"Best params: {best_models[winner]['params']}")
print("========================================")

--- Running GridSearchCV for SVD ---
Best RMSE for SVD: 0.8635485337512646
Best Params for SVD: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}

--- Running GridSearchCV for SVD++ ---
Best RMSE for SVD++: 0.878813754589722
Best Params for SVD++: {'n_factors': 20, 'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.02}

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 1.012936553227082
Best Params for NMF: {'n_factors': 10, 'n_epochs': 15, 'reg_pu': 0.06, 'reg_qi': 0.06}

Best algorithm: SVD
RMSE: 0.8635485337512646
Best params: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


We may conclude after round 1:
- `SVD` might need more factors, epochs, learning rate, regularization - possible underfit since it want everything at max. Best: `0.8635`
- `SVD++` (very slow) might want fewer epochs and factors, everything else is held constant for now.
- `NMF` might need more epochs, hold regularization constant for now

In [16]:
# 1. Create grid search training grid
grids = {
    'SVD': {
        'algo_class': SVD,
        'param_grid': {
            'n_factors': [120, 140],      # k — number of latent factors
            'n_epochs':  [40],       # passes over the data
            'lr_all':    [0.01, 0.05],  # α — learning rate
            'reg_all':   [0.1, 0.2],    # λ — regularization
        },
    },
    'SVD++': {
        'algo_class': SVDpp,
        'param_grid': {
            'n_factors': [18],
            'n_epochs': [7],
            'lr_all':    [0.005],
            'reg_all':   [0.02],
        },
    },
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [10, 20],       # NMF usually requires fewer factors
            'n_epochs': [25],
            # NMF has a separate regularization for users and items
            'reg_pu':    [0.06],
            'reg_qi':    [0.06],
        },
    },
}

best_models = {}

# 2. Run grid search
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=3,
        n_jobs=n_jobs,
    )
    gs.fit(data)

    # Save results
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Best algorithm
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"Best algorithm: {winner}")
print(f"RMSE: {best_models[winner]['score']}")
print(f"Best params: {best_models[winner]['params']}")
print("========================================")

--- Running GridSearchCV for SVD ---
Best RMSE for SVD: 0.8599395992780314
Best Params for SVD: {'n_factors': 140, 'n_epochs': 40, 'lr_all': 0.01, 'reg_all': 0.1}

--- Running GridSearchCV for SVD++ ---
Best RMSE for SVD++: 0.886191614436599
Best Params for SVD++: {'n_factors': 18, 'n_epochs': 7, 'lr_all': 0.005, 'reg_all': 0.02}

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 0.968854137474131
Best Params for NMF: {'n_factors': 10, 'n_epochs': 25, 'reg_pu': 0.06, 'reg_qi': 0.06}

Best algorithm: SVD
RMSE: 0.8599395992780314
Best params: {'n_factors': 140, 'n_epochs': 40, 'lr_all': 0.01, 'reg_all': 0.1}


We may conclude after round 2:
- `SVD` improved. Preferred previous regularization with new higher factors and epochs. New best: `0.8599`. We're going to increase them further having fixed the regularization and learning rate.
- `SVD++` declined. The previous scores preferred, tunning lr and reg now.
- `NMF` improved. Still prefers fewer factors and more epochs. Increasing epochs further.

In [17]:
# 1. Create grid search training grid
grids = {
    'SVD': {
        'algo_class': SVD,
        'param_grid': {
            'n_factors': [160, 180],      # k — number of latent factors
            'n_epochs':  [40],       # passes over the data
            'lr_all':    [0.01],  # α — learning rate
            'reg_all':   [0.1],    # λ — regularization
        },
    },
    'SVD++': {
        'algo_class': SVDpp,
        'param_grid': {
            'n_factors': [20],
            'n_epochs': [10],
            'lr_all':    [0.005, 0.05],
            'reg_all':   [0.02, 0.1],
        },
    },
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [10],       # NMF usually requires fewer factors
            'n_epochs': [35],
            # NMF has a separate regularization for users and items
            'reg_pu':    [0.06],
            'reg_qi':    [0.06],
        },
    },
}

best_models = {}

# 2. Run grid search
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=3,
        n_jobs=n_jobs,
    )
    gs.fit(data)

    # Save results
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Best algorithm
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"Best algorithm: {winner}")
print(f"RMSE: {best_models[winner]['score']}")
print(f"Best params: {best_models[winner]['params']}")
print("========================================")

--- Running GridSearchCV for SVD ---
Best RMSE for SVD: 0.8587852839294602
Best Params for SVD: {'n_factors': 160, 'n_epochs': 40, 'lr_all': 0.01, 'reg_all': 0.1}

--- Running GridSearchCV for SVD++ ---
Best RMSE for SVD++: 0.86981460109389
Best Params for SVD++: {'n_factors': 20, 'n_epochs': 10, 'lr_all': 0.05, 'reg_all': 0.1}

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 0.9485165651133859
Best Params for NMF: {'n_factors': 10, 'n_epochs': 35, 'reg_pu': 0.06, 'reg_qi': 0.06}

Best algorithm: SVD
RMSE: 0.8587852839294602
Best params: {'n_factors': 160, 'n_epochs': 40, 'lr_all': 0.01, 'reg_all': 0.1}


We may conclude after round 3:
- `SVD` improved. Finally stopped at the factors around `160`. Tuning regularization now. New best: `0.8588`.
- `SVD++` improved. Preferred higher regularization. Going to continue increasing it.
- `NMF` improved. Increasing epochs further.

In [18]:
# 1. Create grid search training grid
grids = {
    'SVD': {
        'algo_class': SVD,
        'param_grid': {
            'n_factors': [160],      # k — number of latent factors
            'n_epochs':  [40],       # passes over the data
            'lr_all':    [0.005, 0.01],  # α — learning rate
            'reg_all':   [0.05, 0.1],    # λ — regularization
        },
    },
    'SVD++': {
        'algo_class': SVDpp,
        'param_grid': {
            'n_factors': [20],
            'n_epochs': [10],
            'lr_all':    [0.005],
            'reg_all':   [0.1, 0.2, 0.3],
        },
    },
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [10],       # NMF usually requires fewer factors
            'n_epochs': [45],
            # NMF has a separate regularization for users and items
            'reg_pu':    [0.06],
            'reg_qi':    [0.06],
        },
    },
}

best_models = {}

# 2. Run grid search
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=3,
        n_jobs=n_jobs,
    )
    gs.fit(data)

    # Save results
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Best algorithm
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"Best algorithm: {winner}")
print(f"RMSE: {best_models[winner]['score']}")
print(f"Best params: {best_models[winner]['params']}")
print("========================================")

--- Running GridSearchCV for SVD ---
Best RMSE for SVD: 0.8589032302025621
Best Params for SVD: {'n_factors': 160, 'n_epochs': 40, 'lr_all': 0.01, 'reg_all': 0.1}

--- Running GridSearchCV for SVD++ ---
Best RMSE for SVD++: 0.883039368316215
Best Params for SVD++: {'n_factors': 20, 'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.1}

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 0.9382266717786839
Best Params for NMF: {'n_factors': 10, 'n_epochs': 45, 'reg_pu': 0.06, 'reg_qi': 0.06}

Best algorithm: SVD
RMSE: 0.8589032302025621
Best params: {'n_factors': 160, 'n_epochs': 40, 'lr_all': 0.01, 'reg_all': 0.1}


We may conclude after round 4:
- `SVD` declined very insignificantly. We keep the previous setting as the best; done with this algorithm.
- `SVD++` declined. The regularization went too high. Trying something in (0.1, 0.2).
- `NMF` improved. Increasing epochs further.

Best RMSE for `SVD`: 0.8587852839294602
Best Params for `SVD`: {'n_factors': 160, 'n_epochs': 40, 'lr_all': 0.01, 'reg_all': 0.1}

In [19]:
# 1. Create grid search training grid
grids = {
    'SVD++': {
        'algo_class': SVDpp,
        'param_grid': {
            'n_factors': [20],
            'n_epochs': [10],
            'lr_all':    [0.005],
            'reg_all':   [0.08, 0.15],
        },
    },
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [10],       # NMF usually requires fewer factors
            'n_epochs': [60],
            # NMF has a separate regularization for users and items
            'reg_pu':    [0.06],
            'reg_qi':    [0.06],
        },
    },
}

best_models = {}

# 2. Run grid search
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=3,
        n_jobs=n_jobs,
    )
    gs.fit(data)

    # Save results
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Best algorithm
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"Best algorithm: {winner}")
print(f"RMSE: {best_models[winner]['score']}")
print(f"Best params: {best_models[winner]['params']}")
print("========================================")

--- Running GridSearchCV for SVD++ ---
Best RMSE for SVD++: 0.8832386451344442
Best Params for SVD++: {'n_factors': 20, 'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.08}

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 0.947414805130351
Best Params for NMF: {'n_factors': 10, 'n_epochs': 60, 'reg_pu': 0.06, 'reg_qi': 0.06}

Best algorithm: SVD++
RMSE: 0.8832386451344442
Best params: {'n_factors': 20, 'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.08}


We may conclude after round 5:
- `SVD++` declined. The best was round 3. Done with it.
- `NMF` improved dramatically. Increasing epochs further.

Best RMSE for SVD++: 0.86981460109389
Best Params for SVD++: {'n_factors': 20, 'n_epochs': 10, 'lr_all': 0.05, 'reg_all': 0.1}

In [20]:
# 1. Create grid search training grid
grids = {
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [10],       # NMF usually requires fewer factors
            'n_epochs': [70, 80, 90, 100],
            # NMF has a separate regularization for users and items
            'reg_pu':    [0.06],
            'reg_qi':    [0.06],
        },
    },
}

best_models = {}

# 2. Run grid search
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=3,
        n_jobs=n_jobs,
    )
    gs.fit(data)

    # Save results
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Best algorithm
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"Best algorithm: {winner}")
print(f"RMSE: {best_models[winner]['score']}")
print(f"Best params: {best_models[winner]['params']}")
print("========================================")

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 0.9455541481425507
Best Params for NMF: {'n_factors': 10, 'n_epochs': 70, 'reg_pu': 0.06, 'reg_qi': 0.06}

Best algorithm: NMF
RMSE: 0.9455541481425507
Best params: {'n_factors': 10, 'n_epochs': 70, 'reg_pu': 0.06, 'reg_qi': 0.06}


We may conclude after round 6:
- `NMF` improved slightly. Finally stopped at 'n_epochs': 70. Tuning the regularization now.

In [21]:
# 1. Create grid search training grid
grids = {
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [10],       # NMF usually requires fewer factors
            'n_epochs': [70],
            # NMF has a separate regularization for users and items
            'reg_pu':    [0.04, 0.08, 0.1],
            'reg_qi':    [0.04, 0.08, 0.1],
        },
    },
}

best_models = {}

# 2. Run grid search
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=3,
        n_jobs=n_jobs,
    )
    gs.fit(data)

    # Save results
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Best algorithm
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"Best algorithm: {winner}")
print(f"RMSE: {best_models[winner]['score']}")
print(f"Best params: {best_models[winner]['params']}")
print("========================================")

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 0.9232488939323299
Best Params for NMF: {'n_factors': 10, 'n_epochs': 70, 'reg_pu': 0.1, 'reg_qi': 0.1}

Best algorithm: NMF
RMSE: 0.9232488939323299
Best params: {'n_factors': 10, 'n_epochs': 70, 'reg_pu': 0.1, 'reg_qi': 0.1}


We may conclude after round 7:
- `NMF` improved. Preferred higher regularization. Going to increase it further.

In [22]:
# 1. Create grid search training grid
grids = {
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [10],       # NMF usually requires fewer factors
            'n_epochs': [70],
            # NMF has a separate regularization for users and items
            'reg_pu':    [0.15, 0.2, 0.25],
            'reg_qi':    [0.15, 0.2, 0.25],
        },
    },
}

best_models = {}

# 2. Run grid search
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=3,
        n_jobs=n_jobs,
    )
    gs.fit(data)

    # Save results
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Best algorithm
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"Best algorithm: {winner}")
print(f"RMSE: {best_models[winner]['score']}")
print(f"Best params: {best_models[winner]['params']}")
print("========================================")

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 0.9149767124620721
Best Params for NMF: {'n_factors': 10, 'n_epochs': 70, 'reg_pu': 0.2, 'reg_qi': 0.15}

Best algorithm: NMF
RMSE: 0.9149767124620721
Best params: {'n_factors': 10, 'n_epochs': 70, 'reg_pu': 0.2, 'reg_qi': 0.15}


We may conclude after round 8:
- `NMF` improved. Done. Didn't surpass SVD.

### Summary
SVD is superior

Best RMSE for SVD: 0.8587852839294602
Best Params for SVD: {'n_factors': 160, 'n_epochs': 40, 'lr_all': 0.01, 'reg_all': 0.1}

Best RMSE for SVD++: 0.86981460109389
Best Params for SVD++: {'n_factors': 20, 'n_epochs': 10, 'lr_all': 0.05, 'reg_all': 0.1}

Best RMSE for NMF: 0.9149767124620721
Best Params for NMF: {'n_factors': 10, 'n_epochs': 70, 'reg_pu': 0.2, 'reg_qi': 0.15}

### Fitting and evaluating the best model

In [23]:
final_algo = SVD(
    n_factors=160,
    n_epochs=40,
    lr_all=0.01,
    reg_all=0.1,
    biased=True,
    random_state=RANDOM_SEED,
)

# 3. Build a trainset out of 100% of your raw data (no more splitting!)
full_data = data.build_full_trainset()

# 4. Train the final model on everything
final_algo.fit(full_data)

In [24]:
evaluate(final_algo, df_train, df_test)

RMSE (train) : 0.6159
RMSE (test) : 0.6180


Closest gap, right before overfitting, we're keeping it.

### Actual recommendations. Testing a random user on every movie they haven't rated

In [ ]:
# user to test
user_id = 1

# rated movies by the user
rated   = ratings_raw.loc[ratings_raw.userId == user_id, 'movieId']

# unrate movies be the user
unrated = movies_raw.loc[~movies_raw.movieId.isin(rated), 'movieId']

# predict unrated movies, sort descending
preds = [(m_id, final_algo.predict(user_id, m_id).est) for m_id in unrated]
preds.sort(key=lambda x: x[1], reverse=True)

for m_id, est in preds[:10]:
    title = movies_raw.loc[movies_raw.movieId == m_id, 'title'].values[0]
    print(f"{est:.2f}  {title}")

5.00  Three Colors: Red (Trois couleurs: Rouge) (1994)
5.00  Shawshank Redemption, The (1994)
5.00  In the Name of the Father (1993)
5.00  Godfather, The (1972)
5.00  Philadelphia Story, The (1940)
5.00  His Girl Friday (1940)
5.00  Secrets & Lies (1996)
5.00  Beautiful Thing (1996)
5.00  Streetcar Named Desire, A (1951)
5.00  Paths of Glory (1957)


## Analytical SVD implementation from scratch

Analytical approach isn't supposed to work here. I wanted to see for myself.

In [29]:
import numpy as np
import pandas as pd


def analytical_SVD(M_df: pd.DataFrame):
    """SVD from scratch via the eigenvalue route:  M = U Σ Vᵀ (thin)."""

    # ---------- STEP 0: Make sure there are no nulls ----------
    # Using sparse matrix with nulls will result into artificial ratings of 0 (didn't like the movie), which is false information.
    # Using the usual triplets: movie_id, user_id, rating won't work either. We need a full matrix for every user and every movies in order to decompose it.
    M = M_df.fillna(0).to_numpy()
    m, n = M.shape

    # ---------- Step 1: collapse M into the SMALLER square, symmetric matrix to compute eigenvectors ----------
    #   MMᵀ (m×m) has the SAME eigenvectors as U
    #   MᵀM (n×n) has the SAME eigenvectors as V
    # We compute whichever family is cheaper, then derive the other in Step 3.
    is_wide = m < n
    square = M @ M.T if is_wide else M.T @ M

    # ---------- Step 2: eigen-decompose the square matrix ----------
    # eigh is purpose-built for real symmetric matrices (which MMᵀ / MᵀM always are):
    # it returns real eigenvalues + orthonormal eigenvectors in one shot, and handles
    # repeated eigenvalues correctly. (On paper this is det(square - λI)=0 then plugging
    # λ back in — but a symbolic solver and float eigenvalues don't mix, so we call eigh.)
    eigenvalues, eigenvectors = np.linalg.eigh(square)

    # eigh gives them smallest-first; flip to SVD convention σ₁ ≥ σ₂ ≥ …
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    # singular values σ = √λ  (clamp tiny negative float noise to 0 before the sqrt)
    sigmas = np.sqrt(np.maximum(0.0, eigenvalues))

    # ---------- Step 2 result: name the family we just got by its ROLE ----------
    # wide  → eigenvectors of MMᵀ are the columns of U
    # tall  → eigenvectors of MᵀM are the columns of V
    if is_wide:
        U = eigenvectors
    else:
        V = eigenvectors

    # ---------- Step 3: recover the OTHER family by passing the first back through M ----------
    # Drop σ = 0 directions: they were crushed to nothing and have no partner vector
    # (this is what makes it the THIN SVD). We build the partner column by column.
    keep = sigmas > 1e-9
    sigmas = sigmas[keep]

    if is_wide:
        U = U[:, keep]
        # have U, want V:  vᵢ = (1/σᵢ) · Mᵀ · uᵢ
        V = np.column_stack([(M.T @ U[:, k]) / sigmas[k] for k in range(len(sigmas))])
    else:
        V = V[:, keep]
        # have V, want U:  uᵢ = (1/σᵢ) · M · vᵢ
        U = np.column_stack([(M @ V[:, k]) / sigmas[k] for k in range(len(sigmas))])

    # --- Step 4: assemble. U and V are already themselves — nothing to remap ---
    Sigma = np.diag(sigmas)
    return U, Sigma, V.T

In [30]:
# =====================================================================
# --- VERIFICATION WITH A REALISTIC MOCK MOVIE TABLE (step by step) ---
# =====================================================================
# 4 Users (rows), 3 Movies (cols). NaN = unwatched "holes".
mock_rating_data = {
    "SciFi_Movie":   [5.0, 4.0, np.nan, 1.0],
    "Action_Movie":  [4.0, np.nan, 2.0, 1.0],
    "Romance_Movie": [np.nan, 1.0, 4.0, 5.0],
}
R_table = pd.DataFrame(mock_rating_data, index=["User_1", "User_2", "User_3", "User_4"])

print("--- Step A: Original sparse table (NaN = hole) ---")
R_table


--- Step A: Original sparse table (NaN = hole) ---


,SciFi_Movie,Action_Movie,Romance_Movie
User_1,5.0,4.0,NaN
User_2,4.0,NaN,1.0
User_3,NaN,2.0,4.0
User_4,1.0,1.0,5.0


In [32]:
print("--- Step B: Run analytical SVD on the zero-filled table ---")
U, Sigma, V_T = analytical_SVD(R_table)
print("U shape:", U.shape, "| Sigma shape:", Sigma.shape, "| Vᵀ shape:", V_T.shape)
print("Singular values (diagonal of Σ):", np.round(np.diag(Sigma), 3))

--- Step B: Run analytical SVD on the zero-filled table ---
U shape: (4, 3) | Sigma shape: (3, 3) | Vᵀ shape: (3, 3)
Singular values (diagonal of Σ): [8.036 5.805 2.594]


In [33]:
print("--- Step C: Reconstruct U @ Σ @ Vᵀ ---")
R_reconstructed = U @ Sigma @ V_T
recon_df = pd.DataFrame(np.round(R_reconstructed, 2),
                        index=R_table.index, columns=R_table.columns)
recon_df

--- Step C: Reconstruct U @ Σ @ Vᵀ ---


,SciFi_Movie,Action_Movie,Romance_Movie
User_1,5.0,4.0,0.0
User_2,4.0,-0.0,1.0
User_3,0.0,2.0,4.0
User_4,1.0,1.0,5.0


In [ ]:
print("\n--- Step D: The point — look ONLY at the hole cells ---")

for user, movie in [("User_3", "SciFi_Movie"),
                    ("User_2", "Action_Movie"),
                    ("User_1", "Romance_Movie")]:
    print(f"  hole ({user}, {movie}) -> reconstructed {recon_df.loc[user, movie]:>5}   (a real prediction should be ~3–5)")


--- Step D: The point — look ONLY at the hole cells ---
  hole (User_3, SciFi_Movie) -> reconstructed   0.0   (a real prediction should be ~3–5)
  hole (User_2, Action_Movie) -> reconstructed  -0.0   (a real prediction should be ~3–5)
  hole (User_1, Romance_Movie) -> reconstructed   0.0   (a real prediction should be ~3–5)


### Conclusion
THe analytical version does work, it is just bound to understand missing values as severe dislikes.

## GD-based SVM implementation from scratch - Task 2

If even doing this task, I wouldn't consider implementing some pseudo version as the task is trying to impose. I'd rather create a real thing.

In [40]:
# Step 1: obtain the sparse pivot grid full of 0s (the initial data where for each user there are values in columns for certain movies and empty cells on the ones they didn't rate)
# Step 2: make a dataframe that only includes existing data - each user having watched and rated the movie (e.g., user_id, movie_id, rating)
# Step 3: make a holdout test
# Step 4: Train

def GD_SVD(
        u_idx: pd.Series, i_idx: pd.Series,                     # user index, item index (manual positional mappings to handle random ids from disrupting indexing)
        ratings: pd.Series, num_users: int, num_items: int,     # train ratings, unique users, unique items
        latent_factors: int = 100,                              # how many components to create to explain a user and an item entity
        learning_rate: float = 0.02,                            # learning rate to update all parameters
        regularization: float = 0.02,                           # regularization term for all parameter updates
        epochs: int = 20,                                       # how many times to loop through train
        random_seed: int = 42,                                  # fixed random seed for reproducibility
        ):

    # ---------- STEP 1: Init the params, convert and rename for convenience ----------
    rng = np.random.default_rng(random_seed)
    P = rng.normal(0, 0.1, (num_users, latent_factors)) # user (P)references
    Q = rng.normal(0, 0.1, (num_items, latent_factors)) # item (Q)ualities
    b_u, b_i = np.zeros(num_users), np.zeros(num_items) # bias user, bias item
    mu = ratings.mean()

    u_idx, i_idx, ratings = np.asarray(u_idx), np.asarray(i_idx), np.asarray(ratings)
    n = len(ratings)

    # ---------- STEP 2: Training loop ----------
    # for each epoch
    for ep in range(epochs):
        # shuffle the train each epoch
        for k in rng.permutation(n):
            u, i, r = u_idx[k], i_idx[k], ratings[k]                    # user_id pos index, item_id pos index, true rating of user_id for item_id

            # copy to preserve the original value (because it takes part in computing both P and Q)
            p_u = P[u].copy()
            q_i = Q[i].copy()

            # forward pass
            r_hat = mu + b_u[u] + b_i[i] + p_u @ q_i

            # error
            e = r - r_hat

            # update the user, the item, the bias
            P[u]  += learning_rate * (e * q_i - regularization * p_u)
            Q[i]  += learning_rate * (e * p_u - regularization * q_i)
            b_u[u] += learning_rate * (e - regularization * b_u[u])
            b_i[i] += learning_rate * (e - regularization * b_i[i])

    return mu, b_u, b_i, P, Q

def predict(df_test, mu, b_u, b_i, P, Q):
    u, i = df_test['u'].to_numpy(), df_test['i'].to_numpy()
    dot = np.sum(P[u] * Q[i], axis=1)   # element row-wise p_u · q_i, then summing across columns
    raw = mu + b_u[u] + b_i[i] + dot
    return np.clip(raw, 0.5, 5.0)       # surprise does this; you weren't

In [41]:
# unique user ids, unique item ids
user_ids, item_ids = ratings_raw['userId'].unique(), ratings_raw['movieId'].unique()

# number of unique users, number of unique items
num_users, num_items = len(user_ids), len(item_ids)

# positional mapping to account for random df indexing {df_index : ordinal index}
user_to_idx = {raw: pos for pos, raw in enumerate(user_ids)}
item_to_idx = {raw: pos for pos, raw in enumerate(item_ids)}

# line up the raw created indexes on a copy
ratings = ratings_raw.copy()
ratings['u'] = ratings['userId'].map(user_to_idx)
ratings['i'] = ratings['movieId'].map(item_to_idx)

In [ ]:
# data preparation: 80% train, 10% validation, 10% test (to replace cross-validation nad grid search)
df_train, df_valid = train_test_split(ratings, test_size=0.1, random_state=RANDOM_SEED)
df_valid, df_test = train_test_split(df_valid, test_size=0.1, random_state=RANDOM_SEED)

# train with the best tuned params above
mu, b_u, b_i, P, Q = GD_SVD(df_train['u'], df_train['i'], df_train['rating'], num_users, num_items, 
                            latent_factors=160, learning_rate=0.01, regularization=0.1, epochs=40)

In [80]:
def evaluate_custom(mu, b_u, b_i, P, Q, df_train, df_test):
    # evaluate
    r_true_train = df_train['rating']
    r_hat_train = predict(df_train, mu, b_u, b_i, P, Q)
    rmse_train = root_mean_squared_error(r_true_train, r_hat_train)

    r_true_test = df_test['rating']
    r_hat_test = predict(df_test, mu, b_u, b_i, P, Q)
    rmse_test = root_mean_squared_error(r_true_test, r_hat_test)

    print(f"RMSE (train) : {rmse_train:.4f}")
    print(f"RMSE (test) : {rmse_test:.4f}")
    print(f"Gap: {np.abs(rmse_test-rmse_train):.5f}")

In [65]:
evaluate_custom(mu, b_u, b_i, P, Q, df_train, df_valid)

RMSE (train) : 0.5833
RMSE (test) : 0.8600


It's an overfit. Surprise must implement things differently. These parameters don't fit. If we stop earlier and the score stabilizes, then the overfit is confirmed.

In [96]:
mu, b_u, b_i, P, Q = GD_SVD(df_train['u'], df_train['i'], df_train['rating'], num_users, num_items, 
                            latent_factors=20, learning_rate=0.005, regularization=0.15, epochs=18)

In [97]:
evaluate_custom(mu, b_u, b_i, P, Q, df_train, df_valid)

RMSE (train) : 0.8284
RMSE (test) : 0.8866
Gap: 0.05823


In [98]:
evaluate_custom(mu, b_u, b_i, P, Q, df_train, df_test)

RMSE (train) : 0.8284
RMSE (test) : 0.8918
Gap: 0.06338


### Conclusion

The surprise's implementation must differ. The same scores cannot be reached. I left out the tuning. Since the function is for education sake only, we're not going to implement either grid/cross-validation search or manipulate the function to improve the scores.

Best custom scores: RMSE (train) : 0.8284 RMSE (test) : 0.8918
Best custom params: {latent_factors=20, learning_rate=0.005, regularization=0.15, epochs=18}